<a href="https://colab.research.google.com/github/llelus/DSA-Project/blob/main/01_data_collection.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
# Her session başında bunu çalıştır
!git clone https://github.com/llelus/DSA-Project.git
%cd DSA-Project
!pip install ccxt -q
from google.colab import userdata
token = userdata.get('GITHUB_TOKEN')
!git remote set-url origin https://{token}@github.com/llelus/DSA-Project.git

Cloning into 'DSA-Project'...
remote: Enumerating objects: 46, done.
remote: Counting objects: 100% (46/46), done.
remote: Compressing objects: 100% (41/41), done.
remote: Total 46 (delta 12), reused 20 (delta 2), pack-reused 0 (from 0)
Receiving objects: 100% (46/46), 40.13 KiB | 642.00 KiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/DSA-Project
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 141.2/141.2 kB 4.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.5/6.5 MB 77.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 59.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 223.8/223.8 kB 14.7 MB/s eta 0:00:00


In [2]:
import requests
import pandas as pd
import ccxt
import time
import ast
import os
from datetime import datetime, timedelta

In [3]:
all_markets = []
today = datetime.utcnow()
yesterday = today - timedelta(days=1)

r = requests.get("https://gamma-api.polymarket.com/markets", params={
    "closed": "true",
    "limit": 100,
    "order": "startDate",
    "ascending": "false",
    "startDateMin": yesterday.strftime("%Y-%m-%d"),
    "startDateMax": today.strftime("%Y-%m-%d"),
})

data = r.json()
btc = [
    m for m in data
    if isinstance(m, dict) and (
        "Bitcoin above" in m.get("question", "")
        or "Bitcoin Up or Down" in m.get("question", "")
    )
]

df_poly_clean = pd.DataFrame(btc)
df_poly_clean["startDate"] = pd.to_datetime(df_poly_clean["startDate"], utc=True)
df_poly_clean["endDate"] = pd.to_datetime(df_poly_clean["endDate"], utc=True)
df_poly_clean["yes_price_open"] = df_poly_clean["outcomePrices"].apply(
    lambda x: float(eval(x)[0]) if isinstance(x, str) else float(x[0])
)
df_poly_clean["result_yes_won"] = df_poly_clean["outcomePrices"].apply(
    lambda x: float(eval(x)[0]) if isinstance(x, str) else float(x[0])
).apply(lambda p: 1 if p == 1.0 else 0)

df_poly_clean = df_poly_clean.drop_duplicates(subset="conditionId").reset_index(drop=True)
print(f"Bugün çekilen unique market: {len(df_poly_clean)}")
print(df_poly_clean["question"].tolist())

/tmp/ipykernel_26947/959870841.py:2: DeprecationWarning: datetime.datetime.utcnow() is deprecated and scheduled for removal in a future version. Use timezone-aware objects to represent datetimes in UTC: datetime.datetime.now(datetime.UTC).
  today = datetime.utcnow()


Bugün çekilen unique market: 40
['Bitcoin above 73,000 on April 10, 5AM ET?', 'Bitcoin above 69,800 on April 10, 5AM ET?', 'Bitcoin above 73,400 on April 10, 5AM ET?', 'Bitcoin above 70,600 on April 10, 5AM ET?', 'Bitcoin above 70,200 on April 10, 5AM ET?', 'Bitcoin above 72,600 on April 10, 5AM ET?', 'Bitcoin above 71,000 on April 10, 5AM ET?', 'Bitcoin above 72,200 on April 10, 5AM ET?', 'Bitcoin above 73,800 on April 10, 5AM ET?', 'Bitcoin above 71,400 on April 10, 5AM ET?', 'Bitcoin above 72,400 on April 10, 4AM ET?', 'Bitcoin above 70,800 on April 10, 4AM ET?', 'Bitcoin above 73,200 on April 10, 4AM ET?', 'Bitcoin above 70,000 on April 10, 4AM ET?', 'Bitcoin above 74,000 on April 10, 4AM ET?', 'Bitcoin above 70,400 on April 10, 4AM ET?', 'Bitcoin above 72,800 on April 10, 4AM ET?', 'Bitcoin above 71,600 on April 10, 4AM ET?', 'Bitcoin above 73,600 on April 10, 4AM ET?', 'Bitcoin above 71,200 on April 10, 4AM ET?', 'Bitcoin above 71,400 on April 10, 3AM ET?', 'Bitcoin above 73,800 

In [4]:
all_price_history = []

for idx, row in df_poly_clean.iterrows():
    token_id = ast.literal_eval(row["clobTokenIds"])[0]

    r = requests.get("https://clob.polymarket.com/prices-history", params={
        "market":   token_id,
        "interval": "max",
        "fidelity": 1
    })

    history = r.json().get("history", [])

    for point in history:
        all_price_history.append({
            "conditionId":    row["conditionId"],
            "question":       row["question"],
            "market_start":   row["startDate"],
            "market_end":     row["endDate"],
            "result_yes_won": row["result_yes_won"],
            "timestamp":      pd.to_datetime(point["t"], unit="s", utc=True),
            "yes_price":      point["p"]
        })

    print(f"{row['question'][:50]}: {len(history)} satır")
    time.sleep(0.3)

df_prices = pd.DataFrame(all_price_history).sort_values("timestamp").reset_index(drop=True)
print(f"\nToplam fiyat noktası: {len(df_prices)}")

Bitcoin above 73,000 on April 10, 5AM ET?: 23 satır
Bitcoin above 69,800 on April 10, 5AM ET?: 22 satır
Bitcoin above 73,400 on April 10, 5AM ET?: 15 satır
Bitcoin above 70,600 on April 10, 5AM ET?: 18 satır
Bitcoin above 70,200 on April 10, 5AM ET?: 15 satır
Bitcoin above 72,600 on April 10, 5AM ET?: 16 satır
Bitcoin above 71,000 on April 10, 5AM ET?: 15 satır
Bitcoin above 72,200 on April 10, 5AM ET?: 21 satır
Bitcoin above 73,800 on April 10, 5AM ET?: 23 satır
Bitcoin above 71,400 on April 10, 5AM ET?: 19 satır
Bitcoin above 72,400 on April 10, 4AM ET?: 14 satır
Bitcoin above 70,800 on April 10, 4AM ET?: 15 satır
Bitcoin above 73,200 on April 10, 4AM ET?: 16 satır
Bitcoin above 70,000 on April 10, 4AM ET?: 17 satır
Bitcoin above 74,000 on April 10, 4AM ET?: 15 satır
Bitcoin above 70,400 on April 10, 4AM ET?: 16 satır
Bitcoin above 72,800 on April 10, 4AM ET?: 22 satır
Bitcoin above 71,600 on April 10, 4AM ET?: 22 satır
Bitcoin above 73,600 on April 10, 4AM ET?: 18 satır
Bitcoin abov

In [5]:
exchange = ccxt.kraken()

start_ts = int(df_prices["timestamp"].min().timestamp() * 1000)

ohlcv = exchange.fetch_ohlcv("BTC/USDT", timeframe="1m", since=start_ts, limit=1000)

df_binance = pd.DataFrame(ohlcv, columns=["timestamp","open","high","low","close","volume"])
df_binance["timestamp"] = pd.to_datetime(df_binance["timestamp"], unit="ms", utc=True)
df_binance = df_binance.sort_values("timestamp").reset_index(drop=True)

print(f"Kraken satır: {len(df_binance)}")
print(f"Tarih: {df_binance['timestamp'].min()} → {df_binance['timestamp'].max()}")

Kraken satır: 450
Tarih: 2026-04-10 04:51:00+00:00 → 2026-04-10 12:20:00+00:00


In [6]:
df_merged = pd.merge_asof(
    df_prices,
    df_binance[["timestamp", "close", "volume"]],
    on="timestamp",
    direction="nearest",
    tolerance=pd.Timedelta("2min")
)
df_merged = df_merged.rename(columns={"close": "btc_price", "volume": "btc_volume"})

print(f"Merged satır: {len(df_merged)}")
print(f"NaN btc_price: {df_merged['btc_price'].isna().sum()}")

Merged satır: 716
NaN btc_price: 0


In [7]:
os.makedirs("data/raw", exist_ok=True)
os.makedirs("data/processed", exist_ok=True)

# Mevcut veriyle birleştir
try:
    df_existing = pd.read_csv("data/processed/merged_dataset.csv")
    df_existing["timestamp"] = pd.to_datetime(df_existing["timestamp"], utc=True)
    df_combined = pd.concat([df_existing, df_merged], ignore_index=True)
    df_combined = df_combined.drop_duplicates(subset=["conditionId", "timestamp"])
    df_combined = df_combined.sort_values("timestamp").reset_index(drop=True)
    print(f"Önceki: {len(df_existing)} → Yeni toplam: {len(df_combined)}")
except FileNotFoundError:
    df_combined = df_merged
    print(f"İlk kayıt: {len(df_combined)} satır")

df_combined.to_csv("data/processed/merged_dataset.csv", index=False)
df_prices.to_csv("data/raw/polymarket_raw.csv", index=False)
df_binance.to_csv("data/raw/kraken_raw.csv", index=False)
print("Kaydedildi.")

Önceki: 279 → Yeni toplam: 995
Kaydedildi.


In [9]:
!git config user.email "kadirnsy@gmail.com"
!git config user.name "llelus"
!git add data/
!git commit -m "add: gunluk veri guncellendi"
!git push

[main 13a8acb] add: gunluk veri guncellendi
 3 files changed, 1898 insertions(+), 664 deletions(-)
 rewrite data/raw/kraken_raw.csv (99%)
 rewrite data/raw/polymarket_raw.csv (99%)
Enumerating objects: 15, done.
Counting objects: 100% (15/15), done.
Delta compression using up to 2 threads
Compressing objects: 100% (8/8), done.
Writing objects: 100% (8/8), 23.34 KiB | 2.33 MiB/s, done.
Total 8 (delta 2), reused 0 (delta 0), pack-reused 0
remote: Resolving deltas: 100% (2/2), completed with 1 local object.
To https://github.com/llelus/DSA-Project.git
   5b1e06f..13a8acb  main -> main
